# WaveSlide - Exploratory Data Analysis

# 0. Load the data

Import the libraries

In [1]:
import pathlib
import os
import json
from pathlib import Path
import pandas as pd

Load the dataset

In [2]:
# Create the paths of images and annotations
images_root = Path("../data/raw/images")
annotations_root = Path("../data/raw/annotations")

# The rows of the dataframe
rows = []

# Loop through all annotation files
for annotation_file in annotations_root.glob("*.json"):

    # Extract class name from filename
    # class1.json -> class1
    class_name = annotation_file.stem

    # Corresponding image folder
    image_dir = images_root / class_name

    # Load JSON annotations
    with open(annotation_file, "r") as f:
        annotations = json.load(f)

    # iterate trhough the annotations
    for image_id, ann in annotations.items():
        # Build path based on the image_id on the annotation
        image_path = image_dir / f"{image_id}.jpg"

            # Check if image exists or not
        if not image_path.exists():
            continue
        # Iterate if there is multiple bounding boxes
        for bbox, label in zip(ann["bboxes"], ann["labels"]):
            rows.append({
                # Each column of the dataframe
                "image_id": image_id,
                "image_path": str(image_path),
                "image_exists": image_path.exists(),
                "label": label,
                "bbox_x": bbox[0],
                "bbox_y": bbox[1],
                "bbox_w": bbox[2],
                "bbox_h": bbox[3],
            })

# Create dataframe
df_raw = pd.DataFrame(rows)

# 1. Dataset Overview

## 1.1. Number of Images and Classes

In [3]:
num_images = df_raw.shape[0]
print(f"Number of samples: {num_images}")

num_classes = len(df_raw["label"].value_counts())
print(f"Number of classes: {num_classes}")

Number of samples: 155478
Number of classes: 19


## 1.2. Preview of the Dataset

In [4]:
df_raw.head()

,image_id,image_path,image_exists,label,bbox_x,bbox_y,bbox_w,bbox_h
0,004d1cba-d098-4c66-ae06-4ea68de6578f,../data/raw/images/peace/004d1cba-d098-4c66-ae...,True,peace,0.358043,0.245964,0.068907,0.112570
1,004d1cba-d098-4c66-ae06-4ea68de6578f,../data/raw/images/peace/004d1cba-d098-4c66-ae...,True,no_gesture,0.663795,0.631473,0.059004,0.086041
2,00a9bb8b-a484-486f-8c9f-a815d21cfb10,../data/raw/images/peace/00a9bb8b-a484-486f-8c...,True,peace,0.174231,0.274253,0.108672,0.140922
3,00a9bb8b-a484-486f-8c9f-a815d21cfb10,../data/raw/images/peace/00a9bb8b-a484-486f-8c...,True,no_gesture,0.691098,0.867971,0.105901,0.131586
4,00c45722-a11f-4d81-8e69-0c6ac081dc88,../data/raw/images/peace/00c45722-a11f-4d81-8e...,True,peace,0.414221,0.400808,0.038281,0.048308


## 2. Verify Dataset Integrity

## 2.1. Check for corrupted files

In [5]:

from pathlib import Path

IMAGES_PATH = Path("data/raw/images")

corrupted = []

for file in IMAGES_PATH.rglob("*"):

    if not file.is_file():
        continue


    # OpenCV returns None if unreadable/corrupted
    if img is None:
        corrupted.append(file)

print(f"\nFound {len(corrupted)} corrupted images\n")

for file in corrupted:
    print(file)


Found 0 corrupted images



## 2.2. Check for duplicated files

In [6]:
from pathlib import Path
import hashlib

IMAGES_PATH = Path("data/raw/images")

hashes = {}
duplicates = []

for file in IMAGES_PATH.rglob("*"):
    if not file.is_file():
        continue
    try:
        # hash raw file bytes
        file_hash = hashlib.md5(file.read_bytes()).hexdigest()

        if file_hash in hashes:
            duplicates.append((
                str(hashes[file_hash]),
                str(file)
            ))

        else:
            hashes[file_hash] = file

    except Exception as e:
        print(f"Error reading {file}: {e}")

print(f"\nFound {len(duplicates)} duplicate pairs\n")



Found 0 duplicate pairs



In [7]:
import cv2
import pandas as pd

# Example dataframe

results = []

for _, row in df_raw.iterrows():

    image_path = row["image_path"]
    label = row["label"]

    img = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)

    # skip corrupted images
    if img is None:
        continue
    
    height, width = img.shape[:2]

    aspect_ratio = width / height

    # channel count
    if len(img.shape) == 2:
        channels = 1
    else:
        channels = img.shape[2]

    resolution = f"{width}x{height}"

    results.append({
        "image_path": image_path,
        "label": label,
        "width": width,
        "height": height,
        "aspect_ratio": aspect_ratio,
        "resolution": resolution,
        "channels": channels
    })

# Convert to dataframe
analysis_df = pd.DataFrame(results)

print(analysis_df.head())

                                          image_path       label  width  \
0  ../data/raw/images/peace/004d1cba-d098-4c66-ae...       peace    384   
1  ../data/raw/images/peace/004d1cba-d098-4c66-ae...  no_gesture    384   
2  ../data/raw/images/peace/00a9bb8b-a484-486f-8c...       peace    384   
3  ../data/raw/images/peace/00a9bb8b-a484-486f-8c...  no_gesture    384   
4  ../data/raw/images/peace/00c45722-a11f-4d81-8e...       peace    384   

   height  aspect_ratio resolution  channels  
0     514      0.747082    384x514         3  
1     514      0.747082    384x514         3  
2     512      0.750000    384x512         3  
3     512      0.750000    384x512         3  
4     512      0.750000    384x512         3  


In [8]:
from pathlib import Path
import cv2

path = Path("../data/raw/images/peace/0a0f4035-c217-4f30-806e-158c11386005.jpg")

print(path.exists())

img = cv2.imread(str(path))

print(img is None)

True
False


In [9]:
!pip install opencv-python

In [10]:
!which python
!python --version

/home/diegomucha/me/programming/university/Image Processing/WaveSlide/.venv/bin/python
Python 3.12.3


In [11]:
!python -m pip show opencv-python

Name: opencv-python
Version: 4.13.0.92
Summary: Wrapper package for OpenCV python bindings.
Home-page: https://github.com/opencv/opencv-python
Author: 
Author-email: 
License: Apache 2.0
Location: /home/diegomucha/me/programming/university/Image Processing/WaveSlide/.venv/lib/python3.12/site-packages
Requires: numpy
Required-by: 


In [12]:
!python -m pip install opencv-python